# Tesina: geometría de moldes de mascarillas CPAP a partir de escaneos faciales 3D

Cuaderno principal. Cada sección corresponde a un paso del calendario (P0–P9) y
se puede volver a correr sin miedo: lo que ya está hecho en Drive se salta.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)`.
Código esperado: **versión 1.3.6** (lo imprime P0.2; viene de `tesina.zip`, no de este cuaderno).

**Regla de privacidad:** nada de lo que este cuaderno guarda en Drive (STL, `.npy`,
manifiesto, figuras de pacientes) se sube a GitHub. Lo único que se sube al proyecto
es el ZIP `resultados_tesina_G64_<fecha>.zip` del paso P8, que trae solo tablas con
identificadores anónimos.

La guía completa está en `README_guia_paso_a_paso.md`.

**Si algo falla:** cada paso escribe su avance y sus errores (con la traza completa y
el paciente en el que ocurrió) en `trabajo/bitacora/bitacora_<fecha>.txt` en tu Drive.
Ese archivo no contiene geometría; súbelo al proyecto si necesitas ayuda.

## P0 · Configuración

Edita solo esta celda. Los valores por defecto suponen esta estructura en tu Drive:

```
MyDrive/
  1. Training/                 ← los 368 pares, como carpeta (también sirve un ZIP; ajusta DATOS)
  198618_tesina/
    tesina.zip       ← el código (lo subes una vez)
    trabajo/                   ← aquí escribe el cuaderno (se crea solo)
```

`MODO_SINTETICO = True` corre todo con pacientes inventados (unos 2 minutos), para
comprobar que el entorno funciona antes de tocar los datos reales.

In [ ]:
# ======================= EDITA AQUÍ =======================
MODO_SINTETICO = False          # True = ensayo con datos inventados, sin tocar los datos reales
DATOS          = "/content/drive/MyDrive/1. Training"   # carpeta o ZIP; si no existe, se busca en MyDrive
ZIP_CODIGO     = "/content/drive/MyDrive/198618_tesina/tesina.zip"
GIT_URL        = ""             # opcional: si el código está en GitHub, pon aquí la URL y se clona
CARPETA_TRABAJO = "/content/drive/MyDrive/198618_tesina/trabajo"
EXTRAER_EN     = "/content/datos_locales"   # copia en el disco local de la VM, NO Drive
G              = 64             # 64 = C0; usa 32 si la GPU se queda sin memoria
PARES_ESPERADOS = 368           # el indexador se detiene si encuentra otro número (ver guía)
EXCLUIR_ANIDADOS = False        # True = descartar carpetas de paciente anidadas (ya no hay)
EPOCAS         = 80             # en T4: ~4 s por época (25 nov); si la época 1 tarda más de 40 s: borra trabajo/corridas/C0_G64 y pon 30
# ==========================================================
import os, sys
EN_COLAB = "google.colab" in sys.modules
if not EN_COLAB:                # ejecución local de prueba: siempre sintético
    MODO_SINTETICO = True
print("Colab:", EN_COLAB, "| sintético:", MODO_SINTETICO, "| G =", G)

In [ ]:
# P0.1 · Montar Drive y revisar la GPU
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
import subprocess
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout or "Sin GPU visible")
except FileNotFoundError:
    print("Sin GPU visible (nvidia-smi no existe). Para P5–P6 cambia el entorno a GPU.")

In [ ]:
# P0.2 · Traer el código e instalar lo que falta (trimesh y rtree; lo demás ya viene en Colab)
import re, shutil, zipfile, pathlib
CODIGO = pathlib.Path("/content/codigo") if EN_COLAB else pathlib.Path(os.getcwd())
if EN_COLAB:
    if GIT_URL:
        if not CODIGO.exists():
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(CODIGO)], check=True)
    else:
        if CODIGO.exists():
            shutil.rmtree(CODIGO)
        zc = pathlib.Path(ZIP_CODIGO)
        if not zc.exists():     # buscarlo en MyDrive (también «tesina (1).zip»)
            md_ = pathlib.Path("/content/drive/MyDrive")
            es_codigo = re.compile(r"^tesina( \(\d+\))?\.zip$")   # tesina.zip o «tesina (1).zip»
            cands = [p for p in list(md_.glob("*.zip")) + list(md_.glob("*/*.zip")) if es_codigo.match(p.name)]
            if not cands:
                raise FileNotFoundError(f"No encuentro {ZIP_CODIGO}: sube tesina.zip a tu Drive.")
            zc = max(cands, key=lambda p: p.stat().st_mtime)
            print("uso el ZIP de código más reciente:", zc)
        with zipfile.ZipFile(zc) as z:
            z.extractall(CODIGO)
    # si el ZIP trae una carpeta envolvente (p. ej. tesina/tesina/), entrar en ella
    if not (CODIGO / "tesina" / "__init__.py").exists():
        sub = [p for p in CODIGO.iterdir() if (p / "tesina" / "__init__.py").exists()]
        if not sub:
            raise FileNotFoundError(f"No encuentro la carpeta del código «tesina» (con __init__.py) en {CODIGO}")
        CODIGO = sub[0]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "trimesh", "rtree"], check=True)
sys.path.insert(0, str(CODIGO))
import importlib, tesina
from tesina import pipeline, synthetic
from tesina.config import Config
from tesina.bitacora import configurar
configurar()                    # mensajes en pantalla desde ya; P0.4 agrega el archivo en Drive
print("código en", CODIGO, "| versión", tesina.__version__)

In [ ]:
# P0.3 · Copiar los datos al disco local de la VM (unos 20 GB de rostros y moldes;
# 15–40 min la primera vez en cada sesión). Acepta la carpeta «1. Training» o un ZIP.
# Del ZIP saca solo rostro y molde; si un archivo viene dañado, avisa cuál y sigue.
# Desde aquí ya se escribe la bitácora en CARPETA_TRABAJO/bitacora/.
# Córrela en cada sesión (unos 5 min). Si una sesión solo entrena (P6) y te la saltas,
# P0.4 sigue igual: P6 usa los .npy que ya están en Drive.
from tesina import index_dataset as idx
if MODO_SINTETICO:
    base = pathlib.Path("/content/sintetico" if EN_COLAB else "./_sintetico_local")
    if base.exists():
        shutil.rmtree(base)
    RAIZ = synthetic.build_archive(base / "archivo", n_pairs=20)
    CARPETA_TRABAJO_EF = str(base / "trabajo")
    PARES_ESPERADOS_EF = 20
else:
    configurar(carpeta=pathlib.Path(CARPETA_TRABAJO) / "bitacora")
    RAIZ = pipeline.preparar_datos(DATOS, EXTRAER_EN)
    CARPETA_TRABAJO_EF = CARPETA_TRABAJO
    DANADOS = pipeline.pacientes_danados(EXTRAER_EN)
    PARES_ESPERADOS_EF = PARES_ESPERADOS - len(DANADOS)
    if DANADOS:
        print(f"AVISO: {len(DANADOS)} paciente(s) con rostro o molde dañado en el ZIP: {', '.join(DANADOS)}.")
        print(f"       P1 esperará {PARES_ESPERADOS_EF} pares en vez de {PARES_ESPERADOS}. Manda esta lista.")
print("raíz del archivo:", RAIZ)
print("contenido:", sorted(p.name for p in pathlib.Path(RAIZ).iterdir())[:10])

In [ ]:
# P0.4 · Objeto de configuración C0 (se guarda en Drive junto con cada resultado)
if "RAIZ" not in globals():     # P0.3 saltada: solo sirve para P6 en adelante
    if MODO_SINTETICO:
        raise RuntimeError("En el modo sintético corre P0.3 antes de P0.4.")
    RAIZ = pathlib.Path(EXTRAER_EN) / "1. Training"
    CARPETA_TRABAJO_EF = CARPETA_TRABAJO
    PARES_ESPERADOS_EF = PARES_ESPERADOS
    print("AVISO: te saltaste P0.3; P1–P4 y P7–P9 necesitan los datos, P6 no.")
cfg = Config(name="C0", data_root=str(RAIZ), work_dir=CARPETA_TRABAJO_EF,
             expect_pairs=PARES_ESPERADOS_EF, exclude_nested=EXCLUIR_ANIDADOS,
             grid=G, epochs=EPOCAS)
if MODO_SINTETICO:   # red y épocas pequeñas para que el ensayo dure poco
    cfg = Config(name="C0", data_root=str(RAIZ), work_dir=CARPETA_TRABAJO_EF,
                 expect_pairs=PARES_ESPERADOS_EF, grid=16, base_channels=8, depth=3,
                 epochs=3, patience=0, eval_samples=3000)
cfg.validate()
from tesina.bitacora import configurar
configurar(cfg)          # mensajes con hora en pantalla y bitácora en trabajo/bitacora/
import pandas as pd
pathlib.Path(cfg.work_dir).mkdir(parents=True, exist_ok=True)
print(cfg)
pipeline.estado(cfg)

## P1 · Indexador

Corre primero la autoprueba (10 verificaciones sobre un archivo inventado) y luego
el archivo real. Escribe en `trabajo/indice/`: `manifest.csv` (pid, pliegue, rutas),
`pretrain_faces.csv`, `template_validation.csv` e `index_report.json`.

**Qué revisar:** el número de pares completos. Debe ser **368** (tu carpeta «1. Training»
ya no tiene carpetas anidadas). Si sale menos, lo más probable es que la subida a Drive
no haya terminado; si sale otro número y ya entendiste por qué, pon `PARES_ESPERADOS`
igual a ese número y vuelve a correr. **Anota el número: es la n de la tesina.**

In [ ]:
rep = pipeline.paso_indice(cfg)
print("pares completos:", rep["totals"]["complete_pairs_face_and_mask"])
print("anidados:", rep["anomalies"]["nested_patient_folders"])

## P2 · Verificaciones A1 y A3

* **A1**: ¿el rostro y el molde (`*_mask.stl`) están en el mismo sistema de coordenadas?
  En el paciente 01543 no lo estaban, así que el código registra el molde contra el
  rostro de forma automática y reporta qué tan bien quedó (residuo e incertidumbre en
  mm) en 20 pacientes; dibuja el perfil de 4. Los registros se guardan en
  `trabajo/registros/` y los reutilizan P3, P4 y P7.
* **A3**: ¿la voxelización representa bien el molde sólido a la resolución elegida?
  Compara volúmenes en 10 pacientes y dibuja cortes de 4.

Tiempo esperado: 10–15 minutos (el registro tarda unos 10–25 s por paciente).

Las figuras quedan en `trabajo/verificaciones/` (son geometría de pacientes: no
salen de Drive). Abre dos o tres y confirma a ojo lo que dicen las tablas.

In [ ]:
a1, a3 = pipeline.paso_verificaciones(cfg, n_a1=20 if not MODO_SINTETICO else 6,
                                      n_a3=10 if not MODO_SINTETICO else 4)
display(pd.DataFrame(a1))
display(pd.DataFrame(a3))

In [ ]:
# Ver las figuras de las verificaciones
from IPython.display import Image, display as _d
for p in sorted((cfg.work / "verificaciones").glob("*.png"))[:8]:
    print(p.name); _d(Image(filename=str(p), width=900))

## P3 · Preprocesamiento en la cuadrícula fija compartida

Para cada paciente: registro del molde al rostro (o reutiliza el de P2), orientación
del rostro con PCA y reglas deterministas (borde abierto del escaneo y perfil de la
nariz), caja calculada solo con el rostro, cascarón del rostro y sólido del molde por
paridad de rayos **sin relleno**, todo en los mismos G³ puntos. Guarda en
`trabajo/voxeles_G64/` los `.npy` y la transformación de cada paciente. Es reanudable:
si Colab se desconecta, vuelve a correr la celda.

Cada paciente corre en un proceso aparte: si uno pide demasiada memoria (por ejemplo,
un molde abierto muy pesado), se detiene solo ese paciente con el error «sin memoria»
y la sesión sigue. Con suficiente RAM (L4) procesa varios pacientes a la vez; la
primera línea dice cuántos.

Tiempo esperado: unos 15–35 s por paciente; con 1 proceso, **2 a 3.5 horas para 367**;
con varios, proporcionalmente menos. Puedes dejarlo corriendo con la pestaña abierta.

In [ ]:
filas = pipeline.paso_preprocesamiento(cfg)
rep3 = pd.read_csv(cfg.vox_dir / "preprocess_report.csv")
cols = [c for c in ["pitch_mm", "box_mm", "mask_outside_box_frac", "mask_volume_ratio", "seconds"] if c in rep3]
print(rep3[cols].describe().round(3))
if "error" in rep3:
    print("pacientes con error:", rep3[rep3.error.notna()][["pid", "error"]].to_string())

## P3b · Exclusiones (regla fijada antes de ver resultados)

Lee el reporte de P3 y deja fuera, sin cambiar los pliegues:
- los pacientes donde P3 falló;
- los moldes que no son una malla cerrada (`MOLDES_ABIERTOS`): su sólido no está definido;
- los moldes que no pasan A1 (`NO_PASAN_A1`): menos del 30 % del frente a menos de 2 mm
  o incertidumbre del registro mayor a 3 mm.

Escribe `trabajo/indice/excluidos.csv` (con IDs, se queda en Drive) y
`exclusiones_resumen.json` (solo conteos, va en el ZIP de resultados). Tarda segundos.
**Anota la cohorte final y los conteos por motivo: es la n que se reporta.**
No cambies la regla después de correr P4.

In [ ]:
import json
MOLDES_ABIERTOS = True
NO_PASAN_A1 = True
exc = pipeline.paso_exclusiones(cfg, moldes_abiertos=MOLDES_ABIERTOS, no_pasan_a1=NO_PASAN_A1)
print(json.dumps(exc, indent=1, ensure_ascii=False))

## P4 · Predictores de referencia B1 y B2 y ruta de evaluación

B1 = molde promedio del pliegue de entrenamiento; B2 = molde del paciente
de entrenamiento con el rostro más parecido. Ambos pasan por la misma ruta que C0:
marching cubes, Taubin, regreso a mm y Chamfer/HD95 por región contra el STL de
AirTec. Tiempo esperado: unos 5–15 s por paciente y método en la evaluación.

In [ ]:
ruta = pipeline.paso_referencias(cfg)
m = pd.read_csv(ruta)
print(m.groupby("method")[["dice", "chamfer_total", "chamfer_frente", "hd95_total"]].median().round(3))

## P5 · Prueba de humo de C0 a 16³

Corre el flujo completo de C0 con la red pequeña, 16³ y 3 épocas sobre 40 pacientes
reales (en la carpeta aparte `trabajo/prueba_humo_16/`). Sirve solo para comprobar
que nada se rompe antes de gastar horas de GPU. **Los números a 16³ no son
resultados.** Unos 5–15 minutos.

In [ ]:
if not MODO_SINTETICO:
    pipeline.paso_prueba_humo(cfg)
else:
    print("En modo sintético la configuración ya es una prueba de humo; se salta P5.")

## P6 · Entrenamiento de C0 por pliegue

Entrena los pliegues que falten, uno tras otro, guardando `last.pt` en Drive al final
de cada época. Si Colab se desconecta, vuelve a correr P0 (puedes saltarte P0.3) y
esta celda: continúa desde la última época guardada.

Con una T4 y 64³, cada época tarda unos 4 s y cada pliegue de 3 a 5 minutos (la
parada temprana lo corta si en 20 épocas no mejora); los cinco, unos 25 minutos
(medido el 25 nov). `MAX_PLIEGUES_EN_ESTA_SESION = None` entrena los cinco seguidos.

Si aparece `CUDA out of memory`: baja `G` a 32 en la configuración y repite P3–P4
(la carpeta `voxeles_G32` es otra, no se mezcla) y anótalo para el brief.

In [ ]:
MAX_PLIEGUES_EN_ESTA_SESION = None   # None = los cinco pliegues seguidos
res6 = pipeline.paso_entrenamiento(cfg, max_pliegues=MAX_PLIEGUES_EN_ESTA_SESION)
pipeline.estado(cfg)

## P7 · Evaluación de C0 y comparación con B1 y B2

Necesita los cinco pliegues terminados y los datos copiados en esta sesión (P0.3). Evalúa C0 por la
misma ruta y calcula, sobre los pacientes de los cinco pliegues de prueba juntos,
Wilcoxon de rangos con signo, tamaño de efecto r, IC bootstrap de la mediana de la
diferencia y corrección de Holm. Unos 30–60 minutos.

Guarda su avance cada 10 pacientes: si la sesión se cae, corre P0 y esta celda otra vez
y sigue donde se quedó. Desde la versión 1.3.6 libera la memoria de cada paciente y
muestra la memoria en uso cada 10 pacientes (debe quedarse por debajo de ~3 GB).

In [ ]:
pruebas = pipeline.paso_evaluacion_c0(cfg)

## P7b · Segunda alternativa medida: suavizado del molde (sin volver a entrenar)

Vuelve a evaluar las mismas predicciones de C0 sin suavizar y con suavizado de
Laplace, y las compara contra Taubin (el de la tabla principal) con Wilcoxon y Holm.
Sirve para la decisión entre alternativas de la tesina. Solo cambian las métricas de
superficie (Chamfer, HD95); Dice e IoU son iguales. Unos 20–40 minutos.

In [ ]:
pruebas_suavizado = pipeline.paso_suavizado(cfg)

## P8 · Plantillas, casos ilustrativos y paquete de resultados

Deriva la plantilla básica de cada paciente, guarda en `trabajo/casos_ilustrativos/`
los STL y figuras del mejor, el mediano y el peor caso (se quedan en Drive), y arma
el ZIP con las tablas anónimas. **Ese ZIP es lo que subes al proyecto.**

In [ ]:
zip_resultados = pipeline.paso_paquete(cfg)
print(open(zip_resultados.with_suffix("") / "resumen_resultados.md", encoding="utf-8").read()[:3000])
if EN_COLAB:
    from google.colab import files
    files.download(str(zip_resultados))

## P9 · Prototipo integrado: un escaneo facial → molde y plantilla

Encadena todos los componentes para un solo paciente y mide el tiempo de cada uno:
cargar el rostro, orientarlo y voxelizarlo, elegir el giro con el rostro promedio,
cargar la U-Net, inferir, generar el molde STL y derivar la plantilla STL.

* Sin cambiar nada, usa el primer paciente de la cohorte con el modelo del pliegue
  que **no** lo vio al entrenar, y reporta su Dice contra el molde de AirTec.
* Para procesar cualquier otro rostro, pon su ruta en `ROSTRO_STL`; entonces usa el
  promedio de los modelos de los pliegues terminados.

Los STL quedan en `trabajo/prototipo/` (son geometría de paciente: no se suben). Los
tiempos entran al ZIP de resultados, que esta celda vuelve a armar al final.
Necesita P6 terminado y los datos en esta sesión (P0.3). Menos de un minuto por rostro.

In [ ]:
ROSTRO_STL = ""        # opcional: ruta a un *_face_scan.stl; vacío = paciente de la cohorte
for pid in [None]:     # puedes poner varios IDs, p. ej. ["01543", "02670"], para medir varios
    r = pipeline.paso_prototipo(cfg, pid=pid, rostro_stl=ROSTRO_STL or None)
    print({k: v for k, v in r.items() if k != "etiqueta"})
zip_resultados = pipeline.paso_paquete(cfg)      # rearma el ZIP con los tiempos del prototipo
if EN_COLAB:
    from google.colab import files
    files.download(str(zip_resultados))